# 03 - DermAI Model Evaluation & Calibration Analysis

This notebook performs comprehensive evaluation on the trained DermAI deep learning model checkpoint (`ml/experiments/best.pt`) using the HAM10000 7-class dermatoscopic skin lesion validation split (`ml/data/splits.csv`).

### Diagnostic Classes (HAM10000)
1. **akiec** - Actinic Keratoses and Intraepithelial Carcinoma
2. **bcc** - Basal Cell Carcinoma
3. **bkl** - Benign Keratosis (Solar Lentigines / Seborrheic Keratoses)
4. **df** - Dermatofibroma
5. **mel** - Melanoma
6. **nv** - Melanocytic Nevi
7. **vasc** - Vascular Lesions (Angiomas, Pyogenic Granulomas)

### Key Evaluation Steps:
1. **Checkpoint & Data Loading**: Load best model checkpoint and validation split.
2. **Validation Inference**: Extract prediction logits and softmax probabilities.
3. **Confusion Matrix**: 7x7 normalized heatmap of actual vs predicted classes.
4. **ROC Curves (One-vs-Rest)**: Multi-class ROC curves and AUC scores.
5. **Temperature Scaling Calibration**: Reliability diagram before and after probability calibration (target ECE < 0.10).
6. **Classification Report**: Precision, Recall, F1-Score, and Support for each diagnostic category.
7. **Export Artifacts**: Save all plots and summary metrics to `ml/experiments/evaluation_plots/`.


In [ ]:
import os
import sys
import json
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    roc_auc_score
)

# Project paths
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / "ml").is_dir():
    pass
elif (PROJECT_ROOT.parent / "ml").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.calibration.temperature_scaling import TemperatureScaler, compute_ece

# Output directory for evaluation plots
PLOTS_DIR = PROJECT_ROOT / "ml" / "experiments" / "evaluation_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# HAM10000 7 Diagnostic Classes
CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
CLASS_LABELS = [
    "Actinic Keratoses (akiec)",
    "Basal Cell Carcinoma (bcc)",
    "Benign Keratosis (bkl)",
    "Dermatofibroma (df)",
    "Melanoma (mel)",
    "Melanocytic Nevi (nv)",
    "Vascular Lesions (vasc)"
]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}

print(f"Project root: {PROJECT_ROOT}")
print(f"Evaluation plots directory: {PLOTS_DIR}")
print(f"Evaluation classes: {CLASS_NAMES}")


## 1. Model Loading
Load the best performing model checkpoint (`ml/experiments/best.pt`).


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference device: {device}")

model_path = PROJECT_ROOT / "ml" / "experiments" / "best.pt"
if not model_path.is_file():
    raise FileNotFoundError(f"Checkpoint not found at: {model_path}")

print(f"Loading checkpoint from: {model_path}")
checkpoint = torch.load(model_path, map_location=device, weights_only=False)

arch = checkpoint.get("model_name") or checkpoint.get("model") or checkpoint.get("arch") or "resnet50"
print(f"Detected architecture: {arch}")

# Instantiate model architecture
model = timm.create_model(arch, pretrained=False, num_classes=len(CLASS_NAMES))
state_dict = checkpoint.get("model_state_dict") or checkpoint.get("state_dict") or checkpoint
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()

num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model successfully loaded: {arch} ({num_params:.2f}M parameters)")


## 2. Dataset & DataLoader Setup
Load `ml/data/splits.csv` and prepare PyTorch `DataLoader` for the validation set (`split == 'val'`).


In [ ]:
splits_path = PROJECT_ROOT / "ml" / "data" / "splits.csv"
df_splits = pd.read_csv(splits_path)

val_df = df_splits[df_splits["split"].str.lower() == "val"].copy().reset_index(drop=True)
print(f"Validation samples: {len(val_df)}")
print("Class breakdown in validation cohort:")
print(val_df["dx"].value_counts())

# Image search roots
image_roots = [
    PROJECT_ROOT / "ml" / "data" / "raw" / "images",
    PROJECT_ROOT / "ml" / "data" / "raw",
    PROJECT_ROOT / "ml" / "data" / "hmnist",
]

# Validation Transforms (Standard 224x224 ImageNet normalization)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SkinLesionValDataset(Dataset):
    def __init__(self, df: pd.DataFrame, roots: List[Path], transform=None):
        self.df = df
        self.roots = roots
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row["image_id"])
        dx = str(row["dx"]).lower().strip()
        label = CLASS_TO_IDX.get(dx, 5)

        img_path = None
        for r in self.roots:
            cand = r / f"{img_id}.jpg"
            if cand.is_file():
                img_path = cand
                break

        if img_path and img_path.is_file():
            img = Image.open(img_path).convert("RGB")
        else:
            # Fallback uniform canvas if image file is missing
            img = Image.new("RGB", (224, 224), color=(128, 128, 128))

        if self.transform:
            img = self.transform(img)

        return img, label, img_id

val_dataset = SkinLesionValDataset(val_df, image_roots, transform=val_transforms)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
print(f"Created validation DataLoader with {len(val_loader)} batches.")


## 3. Validation Inference
Run evaluation across the full validation dataset to collect logits, softmax probabilities, and true labels.


In [ ]:
all_logits = []
all_labels = []
all_preds = []

print("Running validation inference...")
with torch.no_grad():
    for batch_idx, (images, targets, _) in enumerate(val_loader):
        images = images.to(device)
        logits = model(images)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(targets.numpy())

all_logits = np.vstack(all_logits)
all_labels = np.concatenate(all_labels)

# Uncalibrated softmax probabilities
uncalibrated_probs = torch.softmax(torch.tensor(all_logits), dim=-1).numpy()
all_preds = np.argmax(uncalibrated_probs, axis=-1)

val_acc = (all_preds == all_labels).mean()
ece_uncalibrated = compute_ece(uncalibrated_probs, all_labels)
print(f"Validation inference complete!")
print(f"Total Evaluated Samples: {len(all_labels)}")
print(f"Validation Accuracy: {val_acc * 100:.2f}%")
print(f"Pre-calibration ECE: {ece_uncalibrated:.4f}")


## 4. Confusion Matrix
Generate and visualize the 7-class normalized confusion matrix.


In [ ]:
cm = confusion_matrix(all_labels, all_preds, labels=range(len(CLASS_NAMES)))
cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-12)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    cbar=True,
    linewidths=0.5
)
plt.title("DermAI Confusion Matrix (Normalized) - Validation Set", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Predicted Class", fontsize=12, fontweight="bold")
plt.ylabel("True Class", fontsize=12, fontweight="bold")
plt.tight_layout()

cm_path = PLOTS_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
print(f"Saved confusion matrix plot to: {cm_path}")
plt.show()


## 5. ROC Curves & One-vs-Rest AUC Analysis
Compute and display the Receiver Operating Characteristic (ROC) curves for all 7 skin lesion categories.


In [ ]:
from sklearn.preprocessing import label_binarize

# One-hot encode targets
y_val_bin = label_binarize(all_labels, classes=list(range(len(CLASS_NAMES))))

fpr = dict()
tpr = dict()
roc_auc = dict()

for i, class_name in enumerate(CLASS_NAMES):
    fpr[i], tpr[i], _ = roc_curve(y_val_bin[:, i], uncalibrated_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Macro-average AUC
macro_roc_auc = roc_auc_score(y_val_bin, uncalibrated_probs, average="macro")

plt.figure(figsize=(10, 8))
colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628", "#f781bf"]

for i, (color, class_name) in enumerate(zip(colors, CLASS_NAMES)):
    plt.plot(
        fpr[i],
        tpr[i],
        color=color,
        lw=2,
        label=f"{class_name.upper()} (AUC = {roc_auc[i]:.3f})"
    )

plt.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--", label="Chance (AUC = 0.500)")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=12, fontweight="bold")
plt.ylabel("True Positive Rate (Sensitivity)", fontsize=12, fontweight="bold")
plt.title(f"Multi-Class One-vs-Rest ROC Curves (Macro AUC = {macro_roc_auc:.3f})", fontsize=14, fontweight="bold", pad=12)
plt.legend(loc="lower right", fontsize=11, frameon=True)
plt.grid(alpha=0.3)
plt.tight_layout()

roc_path = PLOTS_DIR / "roc_curves.png"
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
print(f"Saved ROC curves plot to: {roc_path}")
plt.show()


## 6. Temperature Scaling Probability Calibration
Calibrate output logits using temperature scaling ($T > 0$) to align confidence scores with empirical accuracy, achieving an Expected Calibration Error (ECE) < 0.10.


In [ ]:
# Instantiate and fit TemperatureScaler
scaler = TemperatureScaler()
optimal_t = scaler.fit(all_logits, all_labels)
calibrated_probs = scaler.transform(all_logits)
ece_before = compute_ece(uncalibrated_probs, all_labels, num_bins=15)
ece_after = compute_ece(calibrated_probs, all_labels, num_bins=15)

print(f"Learned Optimal Temperature T: {optimal_t:.4f}")
print(f"ECE Before Calibration: {ece_before:.4f}")
print(f"ECE After Calibration:  {ece_after:.4f} (Target < 0.10: {'PASSED' if ece_after < 0.10 else 'FAILED'})")

# Save temperature configuration
scaler.save(PROJECT_ROOT / "ml" / "experiments" / "temperature.json")

# Reliability Diagram helper
def get_calibration_curve(probs, labels, num_bins=10):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels).astype(float)

    bin_boundaries = np.linspace(0, 1, num_bins + 1)
    bin_accs = []
    bin_confs = []
    bin_counts = []

    for i in range(num_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        prop_in_bin = np.mean(in_bin)
        bin_counts.append(int(np.sum(in_bin)))
        if prop_in_bin > 0:
            bin_accs.append(np.mean(accuracies[in_bin]))
            bin_confs.append(np.mean(confidences[in_bin]))
        else:
            bin_accs.append(0.0)
            bin_confs.append((bin_boundaries[i] + bin_boundaries[i + 1]) / 2)
    return np.array(bin_confs), np.array(bin_accs), np.array(bin_counts)

conf_pre, acc_pre, counts_pre = get_calibration_curve(uncalibrated_probs, all_labels, num_bins=10)
conf_post, acc_post, counts_post = get_calibration_curve(calibrated_probs, all_labels, num_bins=10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Before Calibration
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect Calibration")
axes[0].plot(conf_pre, acc_pre, marker="o", color="#d95f02", lw=2, label=f"Uncalibrated (ECE = {ece_before:.4f})")
axes[0].set_xlabel("Mean Predicted Confidence", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Empirical Accuracy", fontsize=11, fontweight="bold")
axes[0].set_title(f"Before Calibration (T=1.000, ECE={ece_before:.4f})", fontsize=12, fontweight="bold")
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])
axes[0].grid(alpha=0.3)
axes[0].legend(loc="upper left")

# Subplot 2: After Calibration
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect Calibration")
axes[1].plot(conf_post, acc_post, marker="s", color="#1b9e77", lw=2, label=f"Calibrated (T={optimal_t:.2f}, ECE = {ece_after:.4f})")
axes[1].set_xlabel("Mean Predicted Confidence", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Empirical Accuracy", fontsize=11, fontweight="bold")
axes[1].set_title(f"After Temperature Scaling (T={optimal_t:.3f}, ECE={ece_after:.4f})", fontsize=12, fontweight="bold")
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])
axes[1].grid(alpha=0.3)
axes[1].legend(loc="upper left")

plt.suptitle("DermAI Reliability Diagrams: Before vs After Temperature Scaling", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()

cal_plot_path = PLOTS_DIR / "calibration_curve.png"
plt.savefig(cal_plot_path, dpi=300, bbox_inches="tight")
print(f"Saved calibration curve plot to: {cal_plot_path}")
plt.show()


## 7. Classification Report & Metrics Export
Print detailed per-class precision, recall, and F1-score, and save the evaluation summary JSON.


In [ ]:
report_dict = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

report_text = classification_report(
    all_labels,
    all_preds,
    target_names=CLASS_NAMES,
    zero_division=0
)

print("=" * 65)
print("              DERMAI VALIDATION CLASSIFICATION REPORT")
print("=" * 65)
print(report_text)
print("=" * 65)

# Build summary payload
evaluation_summary = {
    "model_architecture": arch,
    "num_parameters_m": round(num_params, 2),
    "validation_samples": int(len(all_labels)),
    "validation_accuracy": round(float(val_acc), 4),
    "macro_auc": round(float(macro_roc_auc), 4),
    "ece_before_calibration": round(float(ece_before), 4),
    "optimal_temperature_t": round(float(optimal_t), 4),
    "ece_after_calibration": round(float(ece_after), 4),
    "ece_threshold_met": bool(ece_after < 0.10),
    "per_class_auc": {c: round(float(roc_auc[i]), 4) for i, c in enumerate(CLASS_NAMES)},
    "classification_report": report_dict,
}

summary_json_path = PLOTS_DIR / "evaluation_summary.json"
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=2)

print(f"Saved evaluation summary JSON to: {summary_json_path}")
